# 피처 엔지니어링 EDA

`train_cleaned.csv`와 `test_cleaned.csv`에서 최종 피처 파일을 바로 생성한다.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
TRAIN_PATH = DATA_DIR / "train_cleaned.csv"
TEST_PATH = DATA_DIR / "test_cleaned.csv"
TRAIN_OUT = DATA_DIR / "train_eda_revised.csv"
TEST_OUT = DATA_DIR / "test_eda_revised.csv"
MACRO_QUARTER_PATH = DATA_DIR / "macro_variable.csv"
MACRO_MARKET_PATH = DATA_DIR / "market_macro_monthly.csv"

DATE_COLS = ["baseline_create_date", "due_in_date", "clear_date"]
TEXT_DTYPES = {
    "cust_number": "string",
    "business_code": "string",
    "name_customer": "string",
    "cust_payment_terms": "string",
    "cust_payment_terms_grp": "string",
}

LATE_BUSINESS_DAYS = 5
# [기존 코드 - 주석처리] N_RECENT = 5
RECENT_HISTORY_WINDOWS = [5, 10, 20]
TIME_DECAY_LAMBDA = 0.8


## 데이터 로드

고객번호처럼 앞자리 0이 중요할 수 있는 컬럼은 문자열로 읽고, 기존 `target`은 비교용으로 `target_old`에 보관한다.


In [ ]:
def load_invoice_data(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=TEXT_DTYPES)
    if "target" in df.columns:
        df = df.rename(columns={"target": "target_old"})

    # 날짜 컬럼은 이후 영업일 계산에 바로 쓸 수 있게 변환한다.
    for col in DATE_COLS:
        df[col] = pd.to_datetime(df[col], errors="coerce")

    return df


train_eda_revised = load_invoice_data(TRAIN_PATH)
test_eda_revised = load_invoice_data(TEST_PATH)

for name, df in [("train", train_eda_revised), ("test", test_eda_revised)]:
    print(f"{name}: rows={len(df):,}, cols={len(df.columns)}, customers={df['cust_number'].nunique():,}")
    display(df.head(3))



## 기본 피처 생성

만기일과 결제일 사이의 영업일 지연, 새 타깃, 주말 만기 여부, 금액 구간을 만든다.


In [ ]:
def business_days_between(start_dates, end_dates) -> pd.Series:
    start_dates = pd.to_datetime(start_dates, errors="coerce")
    if not isinstance(end_dates, pd.Series):
        end_dates = pd.Series(end_dates, index=start_dates.index)
    end_dates = pd.to_datetime(end_dates, errors="coerce")

    result = pd.Series(np.nan, index=start_dates.index, dtype="float")
    valid = start_dates.notna() & end_dates.notna()
    if valid.any():
        start_np = start_dates.loc[valid].values.astype("datetime64[D]")
        end_np = end_dates.loc[valid].values.astype("datetime64[D]")
        result.loc[valid] = np.busday_count(start_np, end_np)
    return result


def add_business_days_late(df: pd.DataFrame) -> None:
    # 주말을 제외한 실제 지연일을 계산한다.
    df["business_days_late"] = business_days_between(df["due_in_date"], df["clear_date"]).astype(int)


def add_basic_features(train_df: pd.DataFrame, test_df: pd.DataFrame) -> np.ndarray:
    for df in [train_df, test_df]:
        add_business_days_late(df)
        df["target"] = (df["business_days_late"] > LATE_BUSINESS_DAYS).astype(int)
        df["due_weekend_flag"] = df["due_in_date"].dt.weekday.isin([5, 6]).astype(int)

    # 금액 구간은 train 기준으로 만들고 test에는 같은 경계를 적용한다.
    _, raw_bins = pd.qcut(train_df["amount_in_usd"], q=4, labels=False, retbins=True, duplicates="drop")
    amount_bins = np.r_[-np.inf, raw_bins[1:-1], np.inf]
    amount_labels = list(range(len(amount_bins) - 1))

    for df in [train_df, test_df]:
        df["amount_bin"] = pd.cut(
            df["amount_in_usd"],
            bins=amount_bins,
            labels=amount_labels,
            include_lowest=True,
        ).astype("int64")

    return amount_bins


amount_bins = add_basic_features(train_eda_revised, test_eda_revised)
print("Amount bin edges:", amount_bins)
print(pd.DataFrame({
    "train_target_old": train_eda_revised["target_old"].value_counts().sort_index(),
    "train_target": train_eda_revised["target"].value_counts().sort_index(),
    "test_target_old": test_eda_revised["target_old"].value_counts().sort_index(),
    "test_target": test_eda_revised["target"].value_counts().sort_index(),
}).fillna(0).astype(int))



## 고객 이력 기반 피처

현재 인보이스 생성 시점에서 알 수 있는 과거 정보만 사용해 고객별 결제 습관을 만든다.


In [ ]:
def add_business_days(start_dates, n_business_days: int) -> pd.Series:
    start_dates = pd.to_datetime(start_dates, errors="coerce")
    result = pd.Series(pd.NaT, index=start_dates.index)
    valid = start_dates.notna()
    if valid.any():
        start_np = start_dates.loc[valid].values.astype("datetime64[D]")
        result.loc[valid] = pd.to_datetime(np.busday_offset(start_np, n_business_days, roll="forward"))
    return result


def known_history_for_date(customer_history: pd.DataFrame, current_date) -> pd.DataFrame:
    prior = customer_history[customer_history["baseline_create_date"] < current_date].copy()
    if prior.empty:
        return prior

    paid_before_current = prior["clear_date"].notna() & (prior["clear_date"] < current_date)
    unpaid_as_of_current = prior["clear_date"].isna() | (prior["clear_date"] >= current_date)
    unpaid_late_days = business_days_between(prior["due_in_date"], current_date)
    # [기존 코드 - 주석처리] known_late_unpaid = unpaid_as_of_current & (unpaid_late_days >= LATE_BUSINESS_DAYS)
    # [FIXED] 고침: target과 동일하게 > 기준을 적용해 정확히 5영업일 지연 건을 정상으로 판정한다.
    known_late_unpaid = unpaid_as_of_current & (unpaid_late_days > LATE_BUSINESS_DAYS)

    # 이미 결제됐거나 현재 시점에 지연 확정인 과거 인보이스만 사용한다.
    known = prior[paid_before_current | known_late_unpaid].copy()
    if known.empty:
        return known

    paid_late_days = business_days_between(known["due_in_date"], known["clear_date"])
    known_late_unpaid = known_late_unpaid.reindex(known.index).fillna(False)

    known["_known_late_target"] = np.where(
        known_late_unpaid,
        1.0,
        known["target"].astype(float),
    )
    known["_known_late"] = np.where(
        known_late_unpaid,
        1.0,
        # [기존 코드 - 주석처리] (paid_late_days >= LATE_BUSINESS_DAYS).astype(float),
        # [FIXED] 고침: 결제 완료된 과거 건도 target과 동일하게 > 기준으로 연체 여부를 계산한다.
        (paid_late_days > LATE_BUSINESS_DAYS).astype(float),
    )
    known["_days_late_as_of_current"] = np.where(
        known_late_unpaid,
        unpaid_late_days.reindex(known.index),
        paid_late_days,
    )

    # 최근 이력 정렬에 사용할 '지연 여부를 알게 된 날짜'를 만든다.
    known["_known_event_date"] = known["clear_date"]
    known.loc[known_late_unpaid, "_known_event_date"] = add_business_days(
        known.loc[known_late_unpaid, "due_in_date"],
        # [기존 코드 - 주석처리] LATE_BUSINESS_DAYS,
        # [FIXED] 고침: > 5 기준에서는 6영업일째부터 연체 확정이므로 +1을 더한다.
        LATE_BUSINESS_DAYS + 1,
    )
    return known


def weighted_late_rate(known: pd.DataFrame, current_date) -> float:
    event_month = known["_known_event_date"].dt.to_period("M")
    current_month = pd.Period(current_date, freq="M")
    month_gap = np.array([current_month.ordinal - month.ordinal for month in event_month], dtype=float)
    weights = TIME_DECAY_LAMBDA ** month_gap
    return float(np.average(known["_known_late"].astype(float), weights=weights))


def add_customer_history_features(current_df: pd.DataFrame, history_df: pd.DataFrame) -> pd.DataFrame:
    features = [
        "cust_allowed_pay_days_late_rate_past",
        "ratio_paid_invoices_late_past",
        "avg_days_late_paid_late_past",
        "sum_outstanding_amount_past",
        "recent_5_late_rate",
        "recent_10_late_rate",
        "recent_20_late_rate",
        "late_rate_time_decay_lambda_0_8",
    ]
    current_df[features] = 0.0

    for cust, current_rows_for_customer in current_df.groupby("cust_number", sort=False):
        customer_history = history_df[history_df["cust_number"].eq(cust)]

        for current_date, current_rows in current_rows_for_customer.groupby("baseline_create_date", sort=True):
            known = known_history_for_date(customer_history, current_date)
            if not known.empty:
                rate_by_allowed_days = known.groupby("Allowed_Pay_Days")["_known_late_target"].mean()
                current_df.loc[current_rows.index, "cust_allowed_pay_days_late_rate_past"] = (
                    current_rows["Allowed_Pay_Days"].map(rate_by_allowed_days).fillna(0.0).astype(float)
                )
                current_df.loc[current_rows.index, "ratio_paid_invoices_late_past"] = float(known["_known_late"].mean())

                known_late = known[known["_known_late"].eq(1.0)]
                if not known_late.empty:
                    current_df.loc[current_rows.index, "avg_days_late_paid_late_past"] = float(
                        known_late["_days_late_as_of_current"].mean()
                    )

                known_sorted = known.sort_values(["_known_event_date", "baseline_create_date", "due_in_date", "clear_date"])
                for window in RECENT_HISTORY_WINDOWS:
                    current_df.loc[current_rows.index, f"recent_{window}_late_rate"] = float(
                        known_sorted.tail(window)["_known_late"].mean()
                    )
                current_df.loc[current_rows.index, "late_rate_time_decay_lambda_0_8"] = weighted_late_rate(known_sorted, current_date)

            # 현재 시점에 아직 결제되지 않은 과거 인보이스 금액 합계.
            outstanding = customer_history[
                (customer_history["baseline_create_date"] < current_date)
                & (customer_history["clear_date"].isna() | (customer_history["clear_date"] >= current_date))
            ]
            current_df.loc[current_rows.index, "sum_outstanding_amount_past"] = max(
                float(outstanding["amount_in_usd"].sum()),
                0.0,
            )

    return current_df


def first_transaction_late_rate(train_df: pd.DataFrame) -> float:
    train_sorted = train_df.sort_values(["cust_number", "baseline_create_date", "due_in_date", "clear_date"])
    first_rows = train_sorted.groupby("cust_number", sort=False).head(1)
    return float(first_rows["target"].mean())


def add_cleared_customer_features(current_df: pd.DataFrame, history_df: pd.DataFrame, baseline_late_rate: float) -> pd.DataFrame:
    features = [
        "current_transaction_count",
        "cleared_count",
        "is_new_customer",
        "is_last_late",
        "recent_3_late_rate",
    ]
    current_df[features] = 0.0

    for cust, current_rows_for_customer in current_df.groupby("cust_number", sort=False):
        customer_history = history_df[history_df["cust_number"].eq(cust)].copy()

        for current_date, current_rows in current_rows_for_customer.groupby("baseline_create_date", sort=True):
            prior_issued = customer_history[customer_history["baseline_create_date"] < current_date]
            current_df.loc[current_rows.index, "current_transaction_count"] = np.arange(
                len(prior_issued) + 1,
                len(prior_issued) + len(current_rows) + 1,
            )

            prior_cleared = customer_history[
                customer_history["clear_date"].notna() & (customer_history["clear_date"] < current_date)
            ].sort_values(["clear_date", "baseline_create_date", "due_in_date"])

            cleared_count = len(prior_cleared)
            current_df.loc[current_rows.index, "cleared_count"] = cleared_count
            current_df.loc[current_rows.index, "is_new_customer"] = int(cleared_count < 3)

            if prior_cleared.empty:
                current_df.loc[current_rows.index, "is_last_late"] = baseline_late_rate
                current_df.loc[current_rows.index, "recent_3_late_rate"] = baseline_late_rate
            else:
                current_df.loc[current_rows.index, "is_last_late"] = float(prior_cleared["target"].iloc[-1])
                current_df.loc[current_rows.index, "recent_3_late_rate"] = float(prior_cleared["target"].tail(3).mean())

    current_df["current_transaction_count"] = current_df["current_transaction_count"].astype(int)
    current_df["cleared_count"] = current_df["cleared_count"].astype(int)
    current_df["is_new_customer"] = current_df["is_new_customer"].astype(int)
    return current_df


train_eda_revised = add_customer_history_features(train_eda_revised, train_eda_revised)
# [기존 코드 - 주석처리] test_history = pd.concat([train_eda_revised, test_eda_revised], axis=0, ignore_index=True)
# [기존 코드 - 주석처리] test_eda_revised = add_customer_history_features(test_eda_revised, test_history)
# [FIXED] 고침: holdout 평가에서 test clear_date/target이 다른 test 행의 피처에 섞이지 않도록 train 이력만 사용한다.
test_history = train_eda_revised.copy()
test_eda_revised = add_customer_history_features(test_eda_revised, test_history)

new_customer_baseline = first_transaction_late_rate(train_eda_revised)
train_eda_revised = add_cleared_customer_features(train_eda_revised, train_eda_revised, new_customer_baseline)
# [기존 코드 - 주석처리] test_history = pd.concat([train_eda_revised, test_eda_revised], axis=0, ignore_index=True)
# [기존 코드 - 주석처리] test_eda_revised = add_cleared_customer_features(test_eda_revised, test_history, new_customer_baseline)
# [FIXED] 고침: cleared 기반 test 피처도 test 라벨 정보 없이 train 이력만으로 계산한다.
test_history = train_eda_revised.copy()
test_eda_revised = add_cleared_customer_features(test_eda_revised, test_history, new_customer_baseline)

history_features = [
    "cust_allowed_pay_days_late_rate_past",
    "ratio_paid_invoices_late_past",
    "avg_days_late_paid_late_past",
    "sum_outstanding_amount_past",
    "recent_3_late_rate",
    "recent_5_late_rate",
    "recent_10_late_rate",
    "recent_20_late_rate",
    "late_rate_time_decay_lambda_0_8",
    "current_transaction_count",
    "cleared_count",
    "is_new_customer",
    "is_last_late",
]
print(f"신규 고객 기준 연체율: {new_customer_baseline:.4f}")
train_eda_revised[history_features].describe()


## 거시경제 변수 병합

시장 지표와 분기 거시경제 지표를 최종 데이터에 병합한다.

In [ ]:
def add_market_macro_features(train_df: pd.DataFrame, test_df: pd.DataFrame, macro_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    macro = pd.read_csv(macro_path)
    market_cols = [col for col in macro.columns if col != "year_month"]
    drop_cols = ["year_month", *market_cols]

    result = []
    for df in [train_df, test_df]:
        featured = df.drop(columns=[col for col in drop_cols if col in df.columns]).copy()
        featured["year_month"] = featured["baseline_create_date"].dt.to_period("M").astype(str)
        featured = featured.merge(macro, on="year_month", how="left")
        result.append(featured)
    return result[0], result[1]


def add_quarterly_macro_features(train_df: pd.DataFrame, test_df: pd.DataFrame, macro_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    macro = pd.read_csv(macro_path)
    macro["year_quarter"] = macro["year_quarter"].astype(str)
    quarter_cols = [col for col in macro.columns if col not in ["quarter_end_date"]]

    result = []
    for df in [train_df, test_df]:
        featured = df.drop(columns=[col for col in quarter_cols if col in df.columns]).copy()
        featured["year_quarter"] = featured["baseline_create_date"].dt.to_period("Q").astype(str)
        featured = featured.merge(macro[quarter_cols], on="year_quarter", how="left")
        result.append(featured)
    return result[0], result[1]


train_eda_revised, test_eda_revised = add_market_macro_features(train_eda_revised, test_eda_revised, MACRO_MARKET_PATH)
train_eda_revised, test_eda_revised = add_quarterly_macro_features(train_eda_revised, test_eda_revised, MACRO_QUARTER_PATH)

macro_features = [
    "vix", "hy_spread_proxy", "vix_lag1", "vix_lag2", "vix_lag3",
    "hy_spread_proxy_lag1", "hy_spread_proxy_lag2", "hy_spread_proxy_lag3",
    "gdp_growth", "unemployment", "cpi_yoy", "fed_rate", "wti_oil", "dxy", "retail_sales",
]
train_eda_revised[macro_features].describe()


## 저장 및 검증

중간 산출물 없이 최종 파일만 저장하고, 행 수와 주요 컬럼 상태를 확인한다.


In [ ]:
train_eda_revised.to_csv(TRAIN_OUT, index=False)
test_eda_revised.to_csv(TEST_OUT, index=False)

for name, path, df in [
    ("train", TRAIN_OUT, train_eda_revised),
    ("test", TEST_OUT, test_eda_revised),
]:
    print(f"Saved {name}: {path}")
    # [기존 코드 - 주석처리] print(f"rows={len(df):,}, cols={len(df.columns)}, business_day_gap={'business_day_gap' in df.columns}")
    # [FIXED] 고침: 실제 생성 컬럼명은 business_days_late이므로 올바른 컬럼을 검증한다.
    print(f"rows={len(df):,}, cols={len(df.columns)}, business_days_late={'business_days_late' in df.columns}")

